In [9]:
from pathlib import Path
import pandas as pd

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from torch.utils.data import TensorDataset, DataLoader

# Loading Dataset (must be label encoded train dataset!)

In [10]:
data_dir = Path("enr-4320-ms")
if not data_dir.is_dir():
    data_dir = Path("Semi_Seismic_Type") / "enr-4320-ms"

train = pd.read_csv(data_dir / "train_encoded.csv")

# Basic_DNN Schematics

Input features : 6 original numerical features

Hidden layer 1	64 neurons, ReLU

Hidden layer 2 32 neurons, ReLU

Output layer : 5 neurons, Softmax

Optimizer :	Adam

Learning rate :	0.001

Loss function :	Sparse categorical cross-entropy

Batch size : 32

Maximum epochs : 150

Early stopping : Patience = 15

Dropout : None initially

Batch normalization : None initially

Class weighting : None initially

Input preprocessing : StandardScaler

Random seed	: 23

In [11]:
# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------
SEED = 23
BATCH_SIZE = 32
MAX_EPOCHS = 1000
PATIENCE = 15
LEARNING_RATE = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Replace these with your six original numerical feature names.
feature_columns = [
    "feature_1",
    "feature_2",
    "feature_3",
    "feature_4",
    "feature_5",
    "feature_6",
]

class_names = [
    "microseismic",   # 0
    "blasting",       # 1
    "drilling",       # 2
    "scaling",        # 3
    "electric noise" # 4
]

# --------------------------------------------------
# 2. Prepare features and labels
# --------------------------------------------------
import re

# Label must already be encoded as integers 0–4.
target_column = "Label"

# Add identifiers with other names here if needed.
exclude_columns = {target_column, "record_number", "sample_code"}

feature_columns = []
numeric_data = {}

for column in train.columns:
    # Exclude ID, sample_id, eventID, etc.
    normalized_name = re.sub(
        r"([a-z0-9])([A-Z])", r"\1_\2", str(column)
    )
    name_parts = re.split(r"[\s_\-]+", normalized_name.lower())

    if column in exclude_columns or "id" in name_parts:
        continue

    original = train[column]

    if (
        pd.api.types.is_bool_dtype(original)
        or pd.api.types.is_datetime64_any_dtype(original)
        or pd.api.types.is_timedelta64_dtype(original)
    ):
        continue

    converted = pd.to_numeric(original, errors="coerce")

    # Include numeric columns and fully convertible numeric-text columns.
    if original.notna().any() and converted[original.notna()].notna().all():
        feature_columns.append(column)
        numeric_data[column] = converted

print("Input features:", feature_columns)
print("Target:", target_column)

if len(feature_columns) != 6:
    raise ValueError(
        f"Expected 6 numeric input features, found {len(feature_columns)}: "
        f"{feature_columns}"
    )

X = pd.DataFrame(numeric_data).to_numpy(dtype=np.float64)

labels = pd.to_numeric(train[target_column], errors="raise")

if labels.isna().any() or not labels.isin(range(5)).all():
    raise ValueError("Label must contain encoded integers 0–4 without missing values.")

y = labels.to_numpy(dtype=np.int64)

if not np.isfinite(X).all():
    raise ValueError("Input features contain missing or infinite values.")

# 70% training, 15% validation, 15% held-out evaluation.
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=SEED,
)

X_val, X_eval, y_val, y_eval = train_test_split(
    X_remaining, y_remaining,
    test_size=0.50,
    stratify=y_remaining,
    random_state=SEED,
)

# Fit preprocessing only on the training split.
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_eval = scaler.transform(X_eval).astype(np.float32)

# --------------------------------------------------
# 3. Data loaders
# --------------------------------------------------
def make_dataset(features, targets):
    return TensorDataset(
        torch.from_numpy(features),
        torch.from_numpy(targets),
    )


generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    make_dataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
)

val_loader = DataLoader(
    make_dataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

eval_loader = DataLoader(
    make_dataset(X_eval, y_eval),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --------------------------------------------------
# 4. Model: 6 → 64 → 32 → 5
# --------------------------------------------------
class SeismicDNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 5),
        )

    def forward(self, x):
        return self.network(x)


model = SeismicDNN().to(device)

criterion = nn.CrossEntropyLoss()  # No class weighting
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print(model)

# --------------------------------------------------
# 5. Training and validation
# --------------------------------------------------
def run_epoch(loader, training=False):
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.set_grad_enabled(training):
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(features)
            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

            batch_count = targets.size(0)
            total_loss += loss.item() * batch_count
            total_correct += (
                logits.argmax(dim=1) == targets
            ).sum().item()
            total_samples += batch_count

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )


history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
}

best_val_loss = float("inf")
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f} | "
            f"Val accuracy: {val_accuracy:.4f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# Restore weights from the best validation epoch.
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}, validation loss: {best_val_loss:.4f}")

# --------------------------------------------------
# 6. Held-out evaluation and Softmax probabilities
# --------------------------------------------------
model.eval()
probability_batches = []

with torch.no_grad():
    for features, _ in eval_loader:
        logits = model(features.to(device))
        probabilities = torch.softmax(logits, dim=1)
        probability_batches.append(probabilities.cpu().numpy())

eval_probabilities = np.concatenate(probability_batches)
eval_predictions = eval_probabilities.argmax(axis=1)

print(f"\nHeld-out accuracy: {accuracy_score(y_eval, eval_predictions):.4f}")

print(
    classification_report(
        y_eval,
        eval_predictions,
        labels=list(range(5)),
        target_names=class_names,
        zero_division=0,
    )
)

print("Confusion matrix — rows: actual, columns: predicted")
print(confusion_matrix(y_eval, eval_predictions, labels=list(range(5))))

Device: cpu
Input features: ['Duration', 'Rising_time', 'Absolute_amplitude', 'Relative_amplitude', 'Pseudo_energy', 'Pseudo_freq']
Target: Label
SeismicDNN(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=5, bias=True)
  )
)
Epoch   1 | Train loss: 1.3011 | Val loss: 0.9760 | Train accuracy: 0.5711 | Val accuracy: 0.6581
Epoch  10 | Train loss: 0.3927 | Val loss: 0.3840 | Train accuracy: 0.8658 | Val accuracy: 0.8794
Epoch  20 | Train loss: 0.3344 | Val loss: 0.3539 | Train accuracy: 0.8789 | Val accuracy: 0.8755
Epoch  30 | Train loss: 0.3075 | Val loss: 0.3582 | Train accuracy: 0.8887 | Val accuracy: 0.8636
Epoch  40 | Train loss: 0.2928 | Val loss: 0.3266 | Train accuracy: 0.8912 | Val accuracy: 0.8834
Epoch  50 | Train loss: 0.2812 | Val loss: 0.3404 | Train accuracy: 0.8882 | Val accuracy: 0.8755
Epoch  60 | T

# Applying Model to Test Dataset

In [ ]:
# Load test data.
data_dir = Path("enr-4320-ms")
test = pd.read_csv(data_dir / "test.csv")

# Use the same features and order as training.
X_test = (
    test[feature_columns]
    .apply(pd.to_numeric, errors="raise")
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(X_test).all():
    raise ValueError("Test features contain missing or infinite values.")

# Apply the existing scaler; do not fit it again.
X_test = scaler.transform(X_test).astype(np.float32)

test_loader = DataLoader(
    torch.from_numpy(X_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model.eval()
prediction_batches = []

with torch.no_grad():
    for features in test_loader:
        logits = model(features.to(device))
        prediction_batches.append(logits.argmax(dim=1).cpu().numpy())

predicted_labels = np.concatenate(prediction_batches)

label_names = {
    0: "microseismic",
    1: "blasting",
    2: "drilling",
    3: "scaling",
    4: "electric noise",
}

# Class: 0=Microseismic, 1=Blasting, 2=Drilling,
#        3=Scaling, 4=Electrical Noise

output = pd.DataFrame({
    "ID": test["ID"].to_numpy(),
    "Class": predicted_labels.astype(np.int64),
})

output_path = data_dir / "Submission.csv"
output.to_csv(output_path, index=False)

print(output.head())
print(f"Saved to: {output_path}")

   ID  Class
0   1      2
1   2      2
2   3      2
3   4      3
4   5      0
Saved to: enr-4320-ms/Kaggle_Submission_CCW.csv


So we now have baseline MLP for Improvement (Kaggle Accuracy 0.85045)

Now what need is to modify hyperparameters, new MLP Modelling, or maybe Feature Engineering/Importance checkings